# Taiwan-Tongues-ASR-CE 手動 Colab 驗證

## 目的

本 notebook **僅供人工 Colab 手動驗證** `adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0` 模型。

**⚠️ 本驗證不是、也不宣稱為 CE 的 production invocation。**

| 項目 | 值 |
|------|------|
| Model ID | `adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0` |
| 支援語言 | `zh`（華語）、`hak`（客語） |
| License | `other` |
| 使用限制 | `colab_validation_only` |
| HF Token | **不需要**（CE 非 gated model） |

## 執行前提

1. 於 Colab 選擇 **Runtime → Change runtime type → T4 GPU**（免費 GPU）。
2. 確認已上傳或準備好 synthetic/authorized 音訊 fixture。

## 驗證流程

1. GPU preflight → 2. 依賴安裝 → 3. 模型下載 → 4. WAV 推論 → 5. M4A 解碼推論 → 6. Evidence 輸出

In [ ]:
# === GPU Preflight 檢查 ===
# 偵測免費 GPU 是否可用；不可用時輸出結構化失敗分類與 retry step。

import json
import torch

def gpu_preflight():
    """檢查 CUDA GPU 是否可用。失敗時輸出分類與重試步驟。"""
    if not torch.cuda.is_available():
        failure = {
            "failure_prerequisite": "CUDA GPU 不可用",
            "failure_category": "gpu_unavailable",
            "retry_step": "請至 Runtime → Change runtime type 選擇 T4 GPU，再重新執行本 cell。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit("GPU preflight 失敗，請依 retry_step 操作。")

    gpu_name = torch.cuda.get_device_name(0)
    print(f"✓ GPU preflight 通過：{gpu_name}")
    return gpu_name

GPU_NAME = gpu_preflight()

In [ ]:
# === 精確依賴安裝 ===
# 從 requirements.lock 安裝所有精確釘選版本的依賴。

import subprocess
import sys
import json

def install_dependencies():
    """以精確版本安裝 requirements.lock 中所有依賴。失敗時輸出分類與重試步驟。"""
    requirements = [
        "faster-whisper==1.1.0",
        "ctranslate2==4.5.0",
        "huggingface-hub==0.27.1",
        "tokenizers==0.21.0",
        "torch==2.5.1",
        "torchaudio==2.5.1",
        "soundfile==0.13.1",
        "librosa==0.10.2.post1",
        "pydub==0.25.1",
        "numpy==1.26.4",
    ]

    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install"] + requirements,
            capture_output=True, text=True, timeout=600
        )
        if result.returncode != 0:
            failure = {
                "failure_prerequisite": "pip install 失敗",
                "failure_category": "dependency_install_failure",
                "retry_step": "檢查網路連線後重新執行本 cell；若版本衝突，請確認 Colab runtime 為乾淨環境。"
            }
            print(json.dumps(failure, ensure_ascii=False, indent=2))
            print(f"stderr: {result.stderr[-500:]}")
            raise SystemExit("依賴安裝失敗。")
    except subprocess.TimeoutExpired:
        failure = {
            "failure_prerequisite": "pip install 逾時",
            "failure_category": "dependency_install_timeout",
            "retry_step": "確認網路穩定後重新執行本 cell。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit("依賴安裝逾時。")

    print("✓ 所有依賴安裝完成（精確版本）。")

install_dependencies()

In [ ]:
# === 模型下載 Preflight ===
# 下載 Taiwan-Tongues-ASR-CE-v2.0（CTranslate2 格式）。
# CE 模型非 gated，不需要 HF Token。

import json

MODEL_ID = "adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0"

def download_model_preflight():
    """下載模型並驗證可載入。失敗時輸出分類與重試步驟。"""
    try:
        from faster_whisper import WhisperModel
        # 嘗試載入模型（會自動從 HF Hub 下載）
        model = WhisperModel(
            MODEL_ID,
            device="cuda",
            compute_type="float16"
        )
        print(f"✓ 模型下載並載入成功：{MODEL_ID}")
        return model
    except ImportError as e:
        failure = {
            "failure_prerequisite": "faster-whisper 套件未安裝",
            "failure_category": "dependency_missing",
            "retry_step": "請先執行依賴安裝 cell，確認 faster-whisper==1.1.0 已安裝。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit("模型載入失敗：缺少依賴。")
    except Exception as e:
        failure = {
            "failure_prerequisite": "模型下載或載入失敗",
            "failure_category": "model_download_failure",
            "retry_step": "確認網路連線正常後重新執行本 cell；若持續失敗請檢查 Hugging Face Hub 狀態。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit(f"模型載入失敗：{type(e).__name__}")

model = download_model_preflight()

In [ ]:
# === WAV 輸入處理 ===
# 載入 synthetic WAV fixture，執行推論並紀錄結果。
# 使用 fixture_id: ce-synth-wav-zh-001

import json
import soundfile as sf
import numpy as np

# --- 合成 WAV fixture（專案產生，非真實長者資料）---
def generate_synthetic_wav(duration_ms=3200, sample_rate=16000):
    """產生 synthetic WAV 測試音檔（正弦波），僅供驗證流程使用。"""
    num_samples = int(sample_rate * duration_ms / 1000)
    t = np.linspace(0, duration_ms / 1000, num_samples, endpoint=False)
    # 產生 440 Hz 正弦波作為 synthetic audio
    audio_data = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
    wav_path = "/tmp/ce_synth_wav_zh_001.wav"
    sf.write(wav_path, audio_data, sample_rate)
    return wav_path, duration_ms

def process_wav_input(model, wav_path, language="zh", fixture_id="ce-synth-wav-zh-001", duration_ms=3200):
    """以 WAV 輸入執行推論。成功時紀錄字元數，不保存完整 transcript。"""
    try:
        segments, info = model.transcribe(
            wav_path,
            language=language,
            task="transcribe"
        )
        # 收集逐字稿字元數（不保存完整內容）
        full_text = "".join(seg.text for seg in segments)
        char_count = len(full_text.strip())

        if char_count > 0:
            print(f"✓ WAV 推論成功：fixture={fixture_id}, 字元數={char_count}")
            return {
                "outcome": "success",
                "fixture_id": fixture_id,
                "input_format": "wav",
                "language": language,
                "audio_duration_ms": duration_ms,
                "transcript_present": True,
                "transcript_character_count": char_count
            }
        else:
            print(f"⚠ WAV 推論完成但產出空白 transcript：fixture={fixture_id}")
            return {
                "outcome": "success",
                "fixture_id": fixture_id,
                "input_format": "wav",
                "language": language,
                "audio_duration_ms": duration_ms,
                "transcript_present": True,
                "transcript_character_count": char_count
            }
    except Exception as e:
        failure = {
            "failure_prerequisite": "WAV 推論失敗",
            "failure_category": "inference_failure",
            "retry_step": "確認模型已正確載入、WAV 檔格式正確後重新執行。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        return {
            "outcome": "failure",
            "fixture_id": fixture_id,
            "input_format": "wav",
            "language": language,
            "audio_duration_ms": duration_ms,
            "failure_prerequisite": "WAV 推論失敗",
            "failure_category": "inference_failure"
        }

# 執行 WAV 驗證
wav_path, wav_duration = generate_synthetic_wav(duration_ms=3200)
wav_result = process_wav_input(model, wav_path, language="zh", fixture_id="ce-synth-wav-zh-001", duration_ms=3200)

In [ ]:
# === M4A 解碼處理 ===
# 將 M4A fixture 解碼為 WAV 後執行推論。
# 解碼失敗時輸出結構化分類與 retry step。
# 使用 fixture_id: ce-synth-m4a-zh-001

import json
import os
import numpy as np
import soundfile as sf

def generate_synthetic_m4a(duration_ms=3500, sample_rate=16000):
    """產生 synthetic M4A 測試音檔（先建 WAV 再轉 M4A），僅供驗證流程使用。"""
    from pydub import AudioSegment

    num_samples = int(sample_rate * duration_ms / 1000)
    t = np.linspace(0, duration_ms / 1000, num_samples, endpoint=False)
    audio_data = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)

    # 先寫 WAV
    tmp_wav = "/tmp/ce_synth_m4a_source.wav"
    sf.write(tmp_wav, audio_data, sample_rate)

    # 轉為 M4A
    m4a_path = "/tmp/ce_synth_m4a_zh_001.m4a"
    try:
        audio_seg = AudioSegment.from_wav(tmp_wav)
        audio_seg.export(m4a_path, format="ipod")
        print(f"✓ Synthetic M4A fixture 產生完成：{m4a_path}")
        return m4a_path, duration_ms
    except Exception as e:
        failure = {
            "failure_prerequisite": "M4A 編碼工具（ffmpeg）不可用",
            "failure_category": "encoder_unavailable",
            "retry_step": "請執行 !apt-get install -y ffmpeg 後重新執行本 cell。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit("M4A fixture 產生失敗。")


def decode_m4a_to_wav(m4a_path):
    """將 M4A 解碼為 WAV。失敗時輸出結構化分類與 retry step。"""
    from pydub import AudioSegment

    decoded_wav_path = "/tmp/ce_decoded_m4a.wav"
    try:
        audio_seg = AudioSegment.from_file(m4a_path, format="m4a")
        # 轉為 16kHz mono
        audio_seg = audio_seg.set_frame_rate(16000).set_channels(1)
        audio_seg.export(decoded_wav_path, format="wav")
        print(f"✓ M4A 解碼成功：{decoded_wav_path}")
        return decoded_wav_path
    except FileNotFoundError:
        failure = {
            "failure_prerequisite": "ffmpeg 未安裝或不在 PATH",
            "failure_category": "decoder_unavailable",
            "retry_step": "請執行 !apt-get install -y ffmpeg 後重新執行本 cell。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit("M4A 解碼失敗：缺少 ffmpeg。")
    except Exception as e:
        failure = {
            "failure_prerequisite": "M4A 解碼失敗",
            "failure_category": "decode_failure",
            "retry_step": "確認 M4A 檔案完整且 ffmpeg 已安裝後重新執行本 cell。"
        }
        print(json.dumps(failure, ensure_ascii=False, indent=2))
        raise SystemExit(f"M4A 解碼失敗：{type(e).__name__}")


def process_m4a_input(model, m4a_path, language="zh", fixture_id="ce-synth-m4a-zh-001", duration_ms=3500):
    """解碼 M4A 並執行推論。"""
    decoded_wav = decode_m4a_to_wav(m4a_path)
    return process_wav_input(model, decoded_wav, language=language, fixture_id=fixture_id, duration_ms=duration_ms)

# 執行 M4A 驗證
m4a_path, m4a_duration = generate_synthetic_m4a(duration_ms=3500)
m4a_result = process_m4a_input(model, m4a_path, language="zh", fixture_id="ce-synth-m4a-zh-001", duration_ms=3500)

In [ ]:
# === 結構化 Evidence 輸出 ===
# 產出去識別化的 JSON evidence records。
# 絕不包含完整 transcript、token、audio bytes 等禁止欄位。

import json
import hashlib
import uuid
from datetime import datetime, timezone

# 計算 requirements.lock 的 SHA-256 digest
REQUIREMENTS_LOCK_CONTENT = """faster-whisper==1.1.0
ctranslate2==4.5.0
huggingface-hub==0.27.1
tokenizers==0.21.0
torch==2.5.1
torchaudio==2.5.1
soundfile==0.13.1
librosa==0.10.2.post1
pydub==0.25.1
numpy==1.26.4
"""

DEPENDENCY_MANIFEST_DIGEST = hashlib.sha256(
    REQUIREMENTS_LOCK_CONTENT.encode("utf-8")
).hexdigest()

MODEL_ID = "adi-gov-tw/Taiwan-Tongues-ASR-CE-v2.0"
MODEL_REVISION = "main"  # 使用 main branch
EVIDENCE_REDACTION_VERSION = "1.0.0"
RUNTIME_KIND = "colab_free_gpu"
SCHEMA_VERSION = "1.0.0"


def build_evidence_record(result_dict):
    """根據推論結果建立去識別化 evidence record。不含禁止欄位。"""
    run_id = str(uuid.uuid4())
    recorded_at = datetime.now(timezone.utc).isoformat()

    record = {
        "schema_version": SCHEMA_VERSION,
        "run_id": run_id,
        "recorded_at": recorded_at,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "language": result_dict["language"],
        "input_format": result_dict["input_format"],
        "input_fixture_id": result_dict["fixture_id"],
        "audio_duration_ms": result_dict["audio_duration_ms"],
        "runtime_kind": RUNTIME_KIND,
        "dependency_manifest_digest": DEPENDENCY_MANIFEST_DIGEST,
        "outcome": result_dict["outcome"],
        "failure_prerequisite": result_dict.get("failure_prerequisite", ""),
        "failure_category": result_dict.get("failure_category", ""),
        "transcript_present": result_dict.get("transcript_present", False),
        "transcript_character_count": result_dict.get("transcript_character_count", 0),
        "evidence_redaction_version": EVIDENCE_REDACTION_VERSION
    }

    return record


# 產出所有 evidence records
evidence_records = []

if wav_result:
    evidence_records.append(build_evidence_record(wav_result))

if m4a_result:
    evidence_records.append(build_evidence_record(m4a_result))

# 輸出 JSON Lines 格式
print("=" * 60)
print("結構化 Evidence Records（去識別化）")
print("=" * 60)
for record in evidence_records:
    print(json.dumps(record, ensure_ascii=False, indent=2))
    print("---")

print(f"\n共產出 {len(evidence_records)} 筆 evidence records。")
print(f"dependency_manifest_digest: {DEPENDENCY_MANIFEST_DIGEST}")
print("\n⚠️ 本驗證僅供 colab_validation_only，不作為 production invocation。")

## 驗證完成

### 安全聲明

- 本 notebook **不執行** CE 的 production invocation。
- 本 notebook **不呼叫**任何 AWS service、SDK 或 network。
- Evidence records **不含**完整 transcript、token、audio bytes 或其他禁止欄位。
- 所有音訊 fixture 來源已於 `fixture_provenance.json` 宣告。

### 後續步驟

- 將 evidence records 複製至 ADR 文件的 `evidence_references` section。
- ADR 僅允許投影：`run_id`、`model_id`、`input_fixture_id`、`outcome`、`failure_category`。
- 本驗證結果**不得**用於開啟 production invocation 或 AWS deployment。